# 🤖 Salama Insurance — Claims Document Agent (RAG Pipeline)

**End-to-end pipeline:** Parse PDFs → Chunk → Embed → Vector Search Index → RAG Agent → Deploy

### Architecture
| Step | Component | Tool |
|------|-----------|------|
| 1. Parse | Extract text from claim PDFs | `ai_parse_document()` |
| 2. Chunk | Split into overlapping segments | SQL `posexplode` + `sequence` |
| 3. Embed & Index | Create vector search index | Databricks Vector Search + `databricks-bge-large-en` |
| 4. Agent | RAG agent with retriever tool | LangChain + `VectorSearchRetrieverTool` |
| 5. Deploy | Log & serve via Model Serving | MLflow + Unity Catalog |

**Document Types:** Claim Forms, Settlement Notifications, Investigation Reports, Denial Letters  
**Source:** `/Volumes/salama_insurance/salama_silver/claim_documents/`

In [0]:
%pip install -U -qqqq databricks-agents>=0.16.0 "mlflow[databricks]>=2.20.2" databricks-langchain databricks-vectorsearch langchain langchain-community azure-storage-file-datalake
dbutils.library.restartPython()

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG = "salama_insurance"
SCHEMA = "salama_silver"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/claim_documents"

# Tables
PARSED_TABLE = f"{CATALOG}.{SCHEMA}.claim_documents_parsed"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.claim_document_chunks"

# Vector Search
VS_ENDPOINT = "claims_vs_endpoint"
VS_INDEX = f"{CATALOG}.{SCHEMA}.claim_documents_index"
EMBEDDING_MODEL = "databricks-bge-large-en"

# LLM
LLM_ENDPOINT = "databricks-claude-sonnet-4-5"

import os
print(f"✅ Configuration loaded")
print(f"   Volume:    {VOLUME_PATH}")
print(f"   Chunks:    {CHUNKS_TABLE}")
print(f"   VS Index:  {VS_INDEX}")
print(f"   LLM:       {LLM_ENDPOINT}")

# List available documents
files = [f for f in os.listdir(VOLUME_PATH) if f.endswith('.pdf')]
print(f"\n📄 Found {len(files)} PDF documents:")
for f in sorted(files):
    print(f"   {f}")

In [0]:
%sql
CREATE OR REPLACE TABLE salama_insurance.salama_silver.claim_documents_parsed AS
WITH parsed_docs AS (
  SELECT
    path,
    ai_parse_document(content) AS parsed
  FROM READ_FILES(
    '/Volumes/salama_insurance/salama_silver/claim_documents/*.pdf',
    format => 'binaryFile'
  )
)
SELECT
  path,
  regexp_extract(path, '([^/]+)\\.pdf$', 1) AS document_name,
  CASE
    WHEN path LIKE '%claim_form%' THEN 'Claim Form'
    WHEN path LIKE '%settlement%' THEN 'Settlement Notification'
    WHEN path LIKE '%investigation%' THEN 'Investigation Report'
    WHEN path LIKE '%denial%' THEN 'Denial Letter'
    ELSE 'Unknown'
  END AS document_type,
  parsed,
  concat_ws('\n\n',
    transform(
      try_cast(parsed:document:elements AS ARRAY<VARIANT>),
      element -> try_cast(element:content AS STRING)
    )
  ) AS full_text,
  try_cast(parsed:error_status AS STRING) AS error_status
FROM parsed_docs

In [0]:
%sql
SELECT document_type,
  COUNT(*) AS doc_count,
  SUM(CASE WHEN error_status IS NOT NULL THEN 1 ELSE 0 END) AS errors,
  ROUND(AVG(LENGTH(full_text)), 0) AS avg_text_length
FROM salama_insurance.salama_silver.claim_documents_parsed
GROUP BY document_type
ORDER BY doc_count DESC

In [0]:
%sql
CREATE OR REPLACE TABLE salama_insurance.salama_silver.claim_document_chunks AS
WITH chunks AS (
  SELECT
    document_name,
    document_type,
    path AS source_uri,
    full_text,
    posexplode(
      transform(
        sequence(0, greatest(length(full_text) - 1, 0), 400),
        pos -> substring(full_text, pos + 1, 500)
      )
    ) AS (chunk_pos, chunk_text)
  FROM salama_insurance.salama_silver.claim_documents_parsed
  WHERE full_text IS NOT NULL AND LENGTH(full_text) > 0
)
SELECT
  md5(concat(document_name, '_', cast(chunk_pos AS STRING))) AS chunk_id,
  document_name,
  document_type,
  source_uri,
  chunk_pos,
  chunk_text,
  current_timestamp() AS indexed_at
FROM chunks
WHERE LENGTH(TRIM(chunk_text)) > 50

In [0]:
%sql
SELECT document_type,
  COUNT(*) AS chunks,
  ROUND(AVG(LENGTH(chunk_text)), 0) AS avg_chunk_len,
  MIN(LENGTH(chunk_text)) AS min_len,
  MAX(LENGTH(chunk_text)) AS max_len
FROM salama_insurance.salama_silver.claim_document_chunks
GROUP BY document_type
ORDER BY chunks DESC

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

# --- Create endpoint (skip if exists) ---
try:
    ep = vsc.get_endpoint(VS_ENDPOINT)
    print(f"✅ Endpoint '{VS_ENDPOINT}' already exists")
except:
    vsc.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print(f"🔄 Creating endpoint '{VS_ENDPOINT}'...")

# --- Create Delta Sync index with managed embeddings ---
try:
    idx = vsc.get_index(VS_ENDPOINT, VS_INDEX)
    print(f"✅ Index '{VS_INDEX}' already exists")
    # Trigger a sync to pick up latest data
    try:
        idx.sync()
        print("   🔄 Triggered index sync")
    except Exception as e:
        print(f"   (sync note: {e})")
except:
    vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=VS_INDEX,
        source_table_name=CHUNKS_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=EMBEDDING_MODEL,
    )
    print(f"🔄 Creating index '{VS_INDEX}'...")
    print("   This may take a few minutes to sync...")

In [0]:
import time

index = vsc.get_index(VS_ENDPOINT, VS_INDEX)

# Wait for sync to complete
for i in range(30):
    status = index.describe()
    state = status.get("status", {}).get("ready", False)
    if state:
        print(f"✅ Index is ready!")
        break
    msg = status.get("status", {}).get("message", "syncing...")
    print(f"⏳ [{i+1}/30] Waiting for index sync... ({msg})")
    time.sleep(30)
else:
    print("⚠️ Index not ready after 15 minutes. Continue anyway — it may still be syncing.")

# Test similarity search
try:
    results = index.similarity_search(
        query_text="What is the claimed amount?",
        columns=["chunk_id", "document_name", "document_type", "chunk_text"],
        num_results=3,
    )
    print("\n🔍 Test query: 'What is the claimed amount?'")
    for doc in results.get("result", {}).get("data_array", []):
        print(f"  📄 {doc[1]} ({doc[2]})")
        print(f"     {str(doc[3])[:150]}...")
        print()
except Exception as e:
    print(f"⚠️ Test query failed (index may still be syncing): {e}")

In [0]:
import mlflow
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langchain.agents import create_agent

# --- Initialize retriever tool ---
claims_retriever = VectorSearchRetrieverTool(
    index_name=VS_INDEX,
    tool_name="claims_document_search",
    tool_description=(
        "Searches Salama Insurance claim documents including claim submission forms, "
        "settlement notifications, investigation reports, and denial letters. "
        "Use this tool to find information about specific claims, policy details, "
        "claimed amounts, settlement amounts, fraud investigations, denial reasons, "
        "and customer information."
    ),
    num_results=5,
    columns=["chunk_id", "document_name", "document_type", "chunk_text"],
)

# --- Initialize LLM ---
llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.1)

# --- System prompt ---
SYSTEM_PROMPT = """You are a Salama Insurance Claims Assistant AI agent. You help insurance staff
search and analyze claim documents including:
- Claim Submission Forms (policyholder info, policy details, claimed amounts)
- Settlement Notifications (approved amounts, payment details)
- Investigation Reports (fraud scores, investigation findings)
- Denial Letters (rejection reasons, appeal process)

Always cite the document name and type when providing information.
If you cannot find the answer in the retrieved documents, say so clearly.
Format financial amounts with AED currency and proper formatting."""

# --- Build agent with LangGraph ---
agent_executor = create_agent(
    model=llm,
    tools=[claims_retriever],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Claims RAG Agent ready!")
print(f"   LLM: {LLM_ENDPOINT}")
print(f"   Retriever: {VS_INDEX}")
print(f"   Tool: claims_document_search")

In [0]:
test_questions = [
    "What claim forms are available and what are their claimed amounts?",
    "Are there any fraud investigation reports? What were the findings?",
    "Which claims were denied and what were the reasons?",
    "What settlement amounts were paid out?",
]

for q in test_questions:
    print(f"\n{'='*70}")
    print(f"❓ {q}")
    print('='*70)
    try:
        response = agent_executor.invoke({"messages": [("user", q)]})
        # Extract the final AI message
        ai_msgs = [m for m in response["messages"] if hasattr(m, 'type') and m.type == 'ai' and m.content]
        if ai_msgs:
            print(f"\n💬 {ai_msgs[-1].content}")
        else:
            print(f"\n💬 (no response)")
    except Exception as e:
        print(f"\n⚠️ Error: {e}")

In [0]:
import mlflow

# --- Write agent code file (models-from-code approach) ---
agent_code = '''
import mlflow
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langchain.agents import create_agent

mlflow.langchain.autolog()

# Configuration
VS_INDEX = "salama_insurance.salama_silver.claim_documents_index"
LLM_ENDPOINT = "databricks-claude-sonnet-4-5"

# Retriever tool
claims_retriever = VectorSearchRetrieverTool(
    index_name=VS_INDEX,
    tool_name="claims_document_search",
    tool_description=(
        "Searches Salama Insurance claim documents including claim submission forms, "
        "settlement notifications, investigation reports, and denial letters. "
        "Use this tool to find information about specific claims, policy details, "
        "claimed amounts, settlement amounts, fraud investigations, denial reasons, "
        "and customer information."
    ),
    num_results=5,
    columns=["chunk_id", "document_name", "document_type", "chunk_text"],
)

# LLM
llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.1)

# System prompt
SYSTEM_PROMPT = """You are a Salama Insurance Claims Assistant AI agent. You help insurance staff
search and analyze claim documents including:
- Claim Submission Forms (policyholder info, policy details, claimed amounts)
- Settlement Notifications (approved amounts, payment details)
- Investigation Reports (fraud scores, investigation findings)
- Denial Letters (rejection reasons, appeal process)

Always cite the document name and type when providing information.
If you cannot find the answer in the retrieved documents, say so clearly.
Format financial amounts with AED currency and proper formatting."""

# Build agent
agent = create_agent(
    model=llm,
    tools=[claims_retriever],
    system_prompt=SYSTEM_PROMPT,
)

# Set the model (required by MLflow models-from-code)
mlflow.models.set_model(agent)
'''

agent_file = "claims_rag_agent_code.py"
with open(agent_file, "w") as f:
    f.write(agent_code)
print(f"✅ Agent code written to {agent_file}")

# --- Log to MLflow with models-from-code ---
mlflow.set_registry_uri("databricks-uc")

resources = [
    mlflow.models.resources.DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT),
    mlflow.models.resources.DatabricksVectorSearchIndex(index_name=VS_INDEX),
]

input_example = {"messages": [{"role": "user", "content": "What claims were settled?"}]}

with mlflow.start_run(run_name="claims_rag_agent") as run:
    model_info = mlflow.langchain.log_model(
        lc_model=agent_file,
        name="claims_agent",
        input_example=input_example,
        resources=resources,
        registered_model_name=f"{CATALOG}.{SCHEMA}.claims_rag_agent",
    )
    print(f"✅ Agent logged successfully!")
    print(f"   Run ID:    {run.info.run_id}")
    print(f"   Model URI: {model_info.model_uri}")
    print(f"   Registry:  {CATALOG}.{SCHEMA}.claims_rag_agent")
    print(f"\n🚀 Next: Deploy to Model Serving for real-time inference")
    print(f"   Or use: mlflow.pyfunc.load_model('{model_info.model_uri}')")

In [0]:
%sql
-- Unity Catalog function that wraps the claims_rag_agent serving endpoint
-- Allows anyone to call: SELECT salama_insurance.salama_silver.ask_claims_agent('your question')

CREATE OR REPLACE FUNCTION salama_insurance.salama_silver.ask_claims_agent(
  question STRING
  COMMENT 'Natural language question about claim documents'
)
RETURNS STRING
COMMENT 'Searches Salama Insurance claim documents (forms, settlements, investigations, denials) using the Claims RAG Agent. Returns AI-generated answers citing document names and financial amounts in AED.'
RETURN (
  SELECT element_at(
    filter(
      ai_query('claims_rag_agent',
        request => named_struct(
          'messages', array(
            named_struct('role', 'user', 'content', question)
          )
        ),
        returnType => 'STRUCT<messages:ARRAY<STRUCT<content:STRING, type:STRING>>>'
      ).messages,
      m -> m.type = 'ai' AND m.content IS NOT NULL AND length(m.content) > 50
    ),
    -1
  ).content
)

In [0]:
%sql
-- Simple usage: just pass a question string
SELECT salama_insurance.salama_silver.ask_claims_agent(
  'What claims were settled and what amounts were paid?'
) AS answer

## 📖 Unity Catalog Function — Usage Examples

### Simple call
```sql
SELECT salama_insurance.salama_silver.ask_claims_agent('Which claims were denied and why?')
```

### Apply to each row in a table
```sql
SELECT
  claim_id,
  salama_insurance.salama_silver.ask_claims_agent(
    'What documents exist for claim ' || claim_id || '?'
  ) AS doc_summary
FROM salama_insurance.salama_silver.fact_claim
WHERE claim_status = 'UNDER_INVESTIGATION'
LIMIT 5
```

### Use in a Genie Space
Add `salama_insurance.salama_silver.ask_claims_agent` as a **UC function** in your Genie Space configuration. Users can then ask document questions naturally.

### Use in a Dashboard
```sql
SELECT salama_insurance.salama_silver.ask_claims_agent(:user_question) AS answer
```

## 🏗️ Architecture

```
┌────────────────────────────────────────────────────────────────────┐
│  Claim PDFs (UC Volume)                                          │
│  claim_form_*.pdf | settlement_*.pdf | investigation_*.pdf       │
└─────────────────────────────────┬──────────────────────────────────┘
                                │
                     ai_parse_document()
                                │
                ┌───────────────▼────────────────┐
                │  Parsed Text (Delta)     │
                │  claim_documents_parsed  │
                └───────────────┬────────────────┘
                                │
                      SQL Chunking (500 chars)
                                │
                ┌───────────────▼────────────────┐
                │  Chunks (Delta)          │
                │  claim_document_chunks   │
                └───────────────┬────────────────┘
                                │
                  databricks-bge-large-en
                                │
                ┌───────────────▼────────────────┐
                │  Vector Search Index     │
                │  claim_documents_index   │
                └───────────────┬────────────────┘
                                │
                ┌───────────────▼────────────────┐
                │  RAG Agent (LangChain)   │
                │  + VectorSearchTool      │
                │  + Claude Sonnet 4.5     │
                └───────────────┬────────────────┘
                                │
                ┌───────────────▼────────────────┐
                │  MLflow + Model Serving  │
                │  Unity Catalog Registry  │
                └────────────────────────────────┘
```

## 🚀 Next Steps

1. **Deploy to Model Serving** — Create a serving endpoint for real-time chat
2. **Add More Tools** — SQL retriever for live claims data, claim status lookup API
3. **Review App** — Use `mlflow.evaluate()` with the Agent Evaluation framework
4. **Build Chat UI** — Create a Databricks App with Streamlit for interactive querying
5. **Add Guardrails** — Implement PII masking and response validation

---
*Salama Insurance Claims Document Agent — Built with Databricks Agent Framework*